# Significancia estadística y tamaño de efecto en clasificación de ratones

Este notebook reemplaza el análisis original por un flujo reproducible para muestras pequeñas y múltiples estímulos.

## Preguntas que responde

1. **Por prueba individual:** ¿la exactitud balanceada del SVM supera lo esperable por azar para un estímulo y una métrica concretos?
2. **Por metodología:** ¿una métrica (energía, entropía, KL, coactivación o Hurst) separa los grupos en al menos un estímulo, o de forma consistente entre estímulos?
3. **Selección de estímulos:** ¿qué estímulos siguen siendo relevantes después de corregir la multiplicidad?
4. **Magnitud del efecto:** ¿cuán separadas están las distribuciones mediante Cohen *d*, Hedges *g* y medidas complementarias?

La prueba principal es una **permutación de etiquetas que repite completamente la validación cruzada**. Para respetar el diseño factorial 2×2, la edad se permuta dentro de cada genotipo y el genotipo dentro de cada edad. Las permutaciones se sincronizan entre estímulos de una misma metodología, lo que permite una corrección **maxT** y pruebas globales de la metodología.

> Referencia metodológica: Combrisson & Jerbi (2015), *Exceeding chance level by chance: The caveat of theoretical chance levels in brain signal classification and statistical assessment of decoding accuracy*.

> **Identificación de sujetos:** en estas tablas, el identificador del ratón está almacenado en el índice (`MR-0644`, `MR-0645`, etc.). El notebook conserva ese índice y lo copia explícitamente a `subject_id`. La columna `type` se usa solo para inferir genotipo y edad.


In [ ]:
from __future__ import annotations

from dataclasses import dataclass
from pathlib import Path
from typing import Callable, Mapping, Sequence
import json
import math
import re
import time
import warnings

import joblib
import numpy as np
import pandas as pd
from IPython.display import display
from joblib import Parallel, delayed
from scipy.stats import binom
from sklearn.base import clone
from sklearn.covariance import LedoitWolf
from sklearn.impute import SimpleImputer
from sklearn.metrics import accuracy_score, balanced_accuracy_score
from sklearn.model_selection import LeaveOneOut, StratifiedKFold, cross_val_predict
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from statsmodels.stats.multitest import multipletests

warnings.filterwarnings('once')
pd.set_option('display.max_columns', 100)
pd.set_option('display.width', 180)

# -----------------------------------------------------------------------------
# CONFIGURACIÓN PRINCIPAL
# -----------------------------------------------------------------------------
PROJECT_ROOT = Path('.')
OUTPUT_DIR = PROJECT_ROOT / 'statistical_results'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Perfil de trabajo:
#   development -> ejecución rápida para revisar carga, errores y estructura.
#   final       -> resolución inferencial recomendada para el informe final.
RUN_PROFILE = 'development'       # 'development' | 'final'

if RUN_PROFILE == 'development':
    N_PERMUTATIONS = 199
    N_BOOTSTRAP_EFFECT = 500
elif RUN_PROFILE == 'final':
    N_PERMUTATIONS = 9_999
    N_BOOTSTRAP_EFFECT = 2_000
else:
    raise ValueError("RUN_PROFILE debe ser 'development' o 'final'.")

N_JOBS = -1
RANDOM_STATE = 20260713

# Único nivel de significancia utilizado en todo el notebook.
# Las pruebas individuales, maxT, FWER y FDR se evalúan exclusivamente con 0.05.
ALPHA = 0.05

# Las permutaciones se agrupan para reducir el overhead de joblib. Un valor
# entre 25 y 100 suele ser adecuado para este tamaño de problema.
PERMUTATION_BATCH_SIZE = 50
JOBLIB_VERBOSE = 10
CHECKPOINT_COMPRESSION = 3
CHECKPOINT_VERSION = 2

# 'loo' reproduce el esquema leave-one-out. Como análisis de sensibilidad puede
# cambiarse a 'stratified_kfold', que suele tener menor varianza en muestras pequeñas.
CV_SCHEME = 'loo'                 # 'loo' | 'stratified_kfold'
N_SPLITS = 5                      # solo para stratified_kfold

# Modelo inferencial recomendado: hiperparámetros fijados a priori. Si se usan
# modelos guardados, se clona el estimador y se reajusta dentro de cada fold.
MODEL_POLICY = 'fixed'            # 'fixed' | 'saved_template'
SVM_KERNEL = 'linear'
SVM_C = 1.0
SVM_CLASS_WEIGHT = 'balanced'

INCLUDE_SIMPLE_EFFECTS = False
SAVE_NULL_DISTRIBUTIONS = False

# Algunos archivos 2D no contienen el ID real. En ese caso se permite una
# reconstrucción controlada por condition × age × posición dentro del estrato.
ALLOW_POSITIONAL_IDS_2D = True

# Filtros opcionales. Use None para incluir todo o un set de nombres.
SELECTED_STIMULUS_TYPES: set[str] | None = None
SELECTED_ANALYSES: set[str] | None = None
SELECTED_CONTRASTS: set[str] | None = None

FEATURE_GROUP_OVERRIDES: dict[tuple[str, str], dict[str, list[str]]] = {}
SAVED_MODEL_OVERRIDES: dict[tuple[str, str, str, str], str] = {}


## 1. Diseño experimental y familias de hipótesis

La corrección de multiplicidad no debe definirse solo porque los archivos tengan una estructura parecida. La familia principal se define por una **pregunta biológica común**:

- tipo de estímulo (`1D_deterministic`, `1D_stochastic`, `2D_stochastic`),
- contraste (`age_main` o `condition_main`),
- esquema de validación cruzada.

Dentro de esa familia se aplica FDR a todas las combinaciones metodología–estímulo. Adicionalmente, dentro de cada metodología se usa maxT, que conserva la dependencia generada por utilizar los mismos ratones y corrige explícitamente la selección del mejor estímulo.

Los contrastes principales son:

- `age_main`: 3 meses vs. 6 meses, permutando edad **dentro de genotipo**.
- `condition_main`: Wild-Type vs. 5xFAD, permutando genotipo **dentro de edad**.

Esto prueba efectos principales controlando el otro factor. Las comparaciones dentro de cada estrato pueden habilitarse para estudiar posibles interacciones.


### Esquema de archivos 2D

El caso `2D_stochastic` admite ahora un archivo por combinación metodología–estímulo. Los valores `0.4`, `0.7` y `0.9` se combinan por `subject_id`; el análisis maxT usa únicamente los ratones comunes a los tres archivos de una metodología. Para Hurst, cada archivo debe contener `H_x` y `H_y` (o dos columnas numéricas inequívocas).


In [ ]:
# -----------------------------------------------------------------------------
# ESTRUCTURAS DE CONFIGURACIÓN
# -----------------------------------------------------------------------------
@dataclass(frozen=True)
class StimulusSetConfig:
    stimulus_type: str
    base_dir: Path
    model_dir: Path
    expected_stimuli: int
    files: Mapping[str, str | Mapping[str, str]]
    # Para tablas agregadas permite conservar solo las primeras n features,
    # manteniendo siempre subject_id, age, condition y demás metadatos.
    max_feature_columns: int | None = None


@dataclass(frozen=True)
class Contrast:
    name: str
    target: str
    positive_class: str
    negative_class: str
    nuisance: str | None = None
    filters: Mapping[str, str] | None = None


@dataclass(frozen=True)
class FeatureGroup:
    stimulus_id: str
    columns: tuple[str, ...]


STIMULUS_SETS = [
    StimulusSetConfig(
        stimulus_type='1D_deterministic',
        base_dir=PROJECT_ROOT / '1D_ind_sin_analyses',
        model_dir=PROJECT_ROOT / '1D_ind_sin_analyses' / 'models',
        expected_stimuli=12,
        max_feature_columns=12,
        files={
            'energy': 'Energy_level3.json',
            'coactivation': 'Coactiv.json',
            'entropy': 'Entropy.json',
            'kl_divergence': 'kl_div.json',
        },
    ),
    StimulusSetConfig(
        stimulus_type='1D_stochastic',
        base_dir=PROJECT_ROOT / '1D_brownian_analyses',
        model_dir=PROJECT_ROOT / '1D_brownian_analyses' / 'models',
        expected_stimuli=3,
        files={
            'energy': 'Energy_level3.json',
            'coactivation': 'Coactiv.json',
            'entropy': 'Entropy.json',
            'kl_divergence': 'kldiv.json',
            'hurst': 'Hurst_regression.json',
        },
    ),
    StimulusSetConfig(
        stimulus_type='2D_stochastic',
        base_dir=PROJECT_ROOT / '2D_analyses',
        model_dir=PROJECT_ROOT / '2D_analyses' / 'models',
        expected_stimuli=3,
        files={
            'energy': {
                '0.4': 'energy_0.4',
                '0.7': 'energy_0.7',
                '0.9': 'energy_0.9',
            },
            'coactivation': {
                '0.4': 'coactiv_0.4',
                '0.7': 'coactiv_0.7',
                '0.9': 'coactiv_0.9',
            },
            'entropy': {
                '0.4': 'Entropy_0.4',
                '0.7': 'Entropy_0.7',
                '0.9': 'Entropy_0.9',
            },
            'kl_divergence': {
                '0.4': 'kldiv_0.4',
                '0.7': 'kldiv_0.7',
                '0.9': 'kldiv_0.9',
            },
            'hurst': {
                '0.4': 'h_est_h4',
                '0.7': 'h_est_h7',
                '0.9': 'h_est_h9',
            },
        },
    ),
]


def build_contrasts(include_simple_effects: bool = False) -> list[Contrast]:
    contrasts = [
        Contrast(
            name='age_main', target='age', nuisance='condition',
            negative_class='3m', positive_class='6m',
        ),
        Contrast(
            name='condition_main', target='condition', nuisance='age',
            negative_class='Wild-Type', positive_class='5xFAD',
        ),
    ]
    if include_simple_effects:
        contrasts.extend([
            Contrast('condition_at_3m', 'condition', '5xFAD', 'Wild-Type', filters={'age': '3m'}),
            Contrast('condition_at_6m', 'condition', '5xFAD', 'Wild-Type', filters={'age': '6m'}),
            Contrast('age_within_WT', 'age', '6m', '3m', filters={'condition': 'Wild-Type'}),
            Contrast('age_within_5xFAD', 'age', '6m', '3m', filters={'condition': '5xFAD'}),
        ])
    return contrasts


def apply_contrast(df: pd.DataFrame, contrast: Contrast) -> pd.DataFrame:
    """Aplica filtros y conserva las dos clases que forman el contraste."""
    required_columns = {contrast.target}
    if contrast.nuisance is not None:
        required_columns.add(contrast.nuisance)
    if contrast.filters:
        required_columns.update(contrast.filters.keys())

    missing_columns = required_columns.difference(df.columns)
    if missing_columns:
        raise KeyError(
            f'El contraste {contrast.name!r} requiere columnas ausentes: '
            f'{sorted(missing_columns)}.'
        )

    out = df.copy()
    for column, value in (contrast.filters or {}).items():
        out = out.loc[out[column] == value].copy()

    selected_classes = [contrast.negative_class, contrast.positive_class]
    out = out.loc[out[contrast.target].isin(selected_classes)].copy()
    if out.empty:
        raise ValueError(f'El contraste {contrast.name!r} no contiene observaciones.')

    class_counts = out[contrast.target].value_counts()
    missing_classes = [c for c in selected_classes if class_counts.get(c, 0) == 0]
    if missing_classes:
        raise ValueError(
            f'El contraste {contrast.name!r} no contiene ambas clases. '
            f'Ausentes: {missing_classes}; conteos: {class_counts.to_dict()}.'
        )
    return out


CONTRASTS = build_contrasts(INCLUDE_SIMPLE_EFFECTS)


### Identificación de sujetos en JSON

El lector prioriza IDs reales almacenados en el índice o en columnas como `sample_id`, `mouse_id` o `subject_id`. Estas columnas se excluyen de las variables predictoras.

Los archivos 2D actuales pueden no contener ningún ID. Con `ALLOW_POSITIONAL_IDS_2D=True`, solo para el esquema 2D de un archivo por estímulo, se generan IDs controlados como `Wild-Type__3m__001`, usando la posición dentro de cada combinación `condition × age`. El notebook exige que todos los archivos tengan los mismos conteos por estrato y advierte que esta reconstrucción supone el mismo orden de ratones dentro de cada estrato. Para resultados definitivos, se recomienda volver a exportar los JSON con `sample_id`.


In [ ]:
# -----------------------------------------------------------------------------
# IDENTIFICACIÓN SEGURA DE LOS RATONES Y ETIQUETAS
# -----------------------------------------------------------------------------
AGE_PATTERNS = {
    '3m': re.compile(r'(?i)(?:^|[^0-9])3\s*(?:m|mo|month|months|mes|meses)(?:[^0-9]|$)'),
    '6m': re.compile(r'(?i)(?:^|[^0-9])6\s*(?:m|mo|month|months|mes|meses)(?:[^0-9]|$)'),
}
CONDITION_PATTERNS = {
    '5xFAD': re.compile(r'(?i)5\s*x\s*FAD'),
    'Wild-Type': re.compile(r'(?i)(?:wild[\s_-]*type|(?:^|[^A-Za-z])WT(?:[^A-Za-z]|$))'),
}

SUBJECT_ID_CANDIDATES = (
    'subject_id', 'sample_id', 'mouse_id', 'animal_id', 'raton_id', 'ratón_id',
    'mouse', 'animal', 'subject', 'id',
)
GROUP_LABEL_CANDIDATES = ('type', 'group', 'label', 'class')


def normalize_feature_column_names(df: pd.DataFrame) -> pd.DataFrame:
    """Limpia espacios/caracteres invisibles y normaliza aliases de metadatos."""
    out = df.copy()
    aliases = {
        'age': 'age',
        'edad': 'age',
        'age_group': 'age',
        'grupo_edad': 'age',
        'condition': 'condition',
        'condicion': 'condition',
        'condición': 'condition',
        'experimental_condition': 'condition',
        'genotipo': 'condition',
        'genotype': 'condition',
        'type': 'type',
        'tipo': 'type',
        'group': 'group',
        'grupo': 'group',
        'label': 'label',
        'class': 'class',
        'id': 'id',
        'sampleid': 'sample_id',
        'sample_id': 'sample_id',
        'mouseid': 'mouse_id',
        'mouse_id': 'mouse_id',
        'subjectid': 'subject_id',
        'subject_id': 'subject_id',
        'animalid': 'animal_id',
        'animal_id': 'animal_id',
    }

    cleaned_columns: list[str] = []
    for column in out.columns:
        cleaned = str(column).replace('\ufeff', '').replace('\u200b', '').strip()
        key = re.sub(r'[\s\-]+', '_', cleaned.casefold())
        key = re.sub(r'_+', '_', key).strip('_')
        cleaned_columns.append(aliases.get(key, cleaned))

    duplicated = pd.Index(cleaned_columns)[pd.Index(cleaned_columns).duplicated(keep=False)]
    if len(duplicated):
        raise ValueError(
            'La normalización produjo columnas duplicadas: '
            f'{sorted(set(map(str, duplicated)))}. Columnas originales: {list(df.columns)}'
        )
    out.columns = cleaned_columns
    return out


def _match_unique(text: str, patterns: Mapping[str, re.Pattern], variable: str) -> str:
    text = str(text)
    matches = [label for label, pattern in patterns.items() if pattern.search(text)]
    if len(matches) != 1:
        raise ValueError(
            f'No se pudo inferir {variable} de {text!r}. Coincidencias: {matches}. '
            'Ajuste los patrones o agregue columnas explícitas age/condition.'
        )
    return matches[0]


def _is_complete_unique(series: pd.Series) -> bool:
    values = series.astype('string')
    return bool(values.notna().all() and values.str.strip().ne('').all() and values.is_unique)


def _is_default_positional_index(index: pd.Index) -> bool:
    if isinstance(index, pd.RangeIndex):
        return index.start == 0 and index.step == 1 and index.stop == len(index)
    try:
        numeric = pd.to_numeric(pd.Index(index), errors='raise')
        return np.array_equal(np.asarray(numeric), np.arange(len(index)))
    except (TypeError, ValueError):
        return False


def _subject_ids_from_index(df: pd.DataFrame) -> pd.Series | None:
    if not df.index.is_unique or _is_default_positional_index(df.index):
        return None
    values = pd.Series(df.index.map(str), index=df.index, name='subject_id', dtype='string')
    return values if _is_complete_unique(values) else None


def _available_label_source(out: pd.DataFrame) -> pd.Series | None:
    for column in GROUP_LABEL_CANDIDATES:
        if column in out.columns and out[column].notna().all():
            return out[column].astype(str)
    if 'subject_id' in out.columns:
        return out['subject_id'].astype(str)
    return None


def _normalize_age_value(value: object) -> str:
    if pd.isna(value):
        raise ValueError('Se encontró un valor vacío en age.')
    text = str(value).strip()
    compact = re.sub(r'[\s_-]+', '', text).casefold()
    aliases = {
        'young': '3m', 'joven': '3m', 'youngmouse': '3m',
        '3': '3m', '3.0': '3m', '3m': '3m', '3month': '3m', '3months': '3m',
        'old': '6m', 'viejo': '6m', 'oldmouse': '6m',
        '6': '6m', '6.0': '6m', '6m': '6m', '6month': '6m', '6months': '6m',
    }
    if compact in aliases:
        return aliases[compact]
    return _match_unique(text, AGE_PATTERNS, 'age')


def _normalize_condition_value(value: object) -> str:
    if pd.isna(value):
        raise ValueError('Se encontró un valor vacío en condition.')
    text = str(value).strip()
    compact = re.sub(r'[\s_-]+', '', text).casefold()
    aliases = {
        'wt': 'Wild-Type', 'wildtype': 'Wild-Type', 'wild': 'Wild-Type',
        '5xfad': '5xFAD', '5fad': '5xFAD',
    }
    if compact in aliases:
        return aliases[compact]
    return _match_unique(text, CONDITION_PATTERNS, 'condition')


def _controlled_positional_ids(out: pd.DataFrame) -> pd.Series:
    strata = out['condition'].astype(str) + '__' + out['age'].astype(str)
    order = out.groupby(['condition', 'age'], sort=False).cumcount() + 1
    return strata + '__' + order.astype(str).str.zfill(3)


def add_subject_metadata(
    df: pd.DataFrame,
    *,
    allow_positional_ids: bool = False,
) -> pd.DataFrame:
    """Agrega subject_id, age y condition con validaciones de unicidad."""
    out = df.copy()

    index_ids = _subject_ids_from_index(out)
    if index_ids is not None:
        out['subject_id'] = index_ids.to_numpy(dtype=str)
        id_source = 'índice original'
    else:
        id_source = None
        for column in SUBJECT_ID_CANDIDATES:
            if column in out.columns and _is_complete_unique(out[column]):
                out['subject_id'] = out[column].astype(str).to_numpy()
                id_source = f'columna {column!r}'
                break

    label_source = _available_label_source(out)

    age_column = next(
        (c for c in ('age', 'edad', 'age_group', 'grupo_edad') if c in out.columns),
        None,
    )
    if age_column is not None:
        out['age'] = out[age_column].map(_normalize_age_value)
    elif label_source is not None:
        out['age'] = label_source.map(lambda value: _match_unique(value, AGE_PATTERNS, 'age'))
    else:
        raise ValueError(
            'No se encontró una columna age ni una etiqueta grupal desde la cual inferirla. '
            f'Columnas disponibles: {list(out.columns)}'
        )

    condition_column = next(
        (c for c in ('condition', 'genotype', 'condicion', 'condición', 'genotipo') if c in out.columns),
        None,
    )
    if condition_column is not None:
        out['condition'] = out[condition_column].map(_normalize_condition_value)
    elif label_source is not None:
        out['condition'] = label_source.map(
            lambda value: _match_unique(value, CONDITION_PATTERNS, 'condition')
        )
    else:
        raise ValueError(
            'No se encontró una columna condition/genotype ni una etiqueta grupal desde la cual '
            f'inferirla. Columnas disponibles: {list(out.columns)}'
        )

    if id_source is None:
        if not allow_positional_ids:
            raise ValueError(
                'No se encontró un identificador único por ratón. Se esperaba un índice como '
                'MR-0644 o una columna explícita sample_id/mouse_id/subject_id. '
                f'Índice observado: {out.index[:5].tolist()}; columnas: {out.columns.tolist()}.'
            )
        out['subject_id'] = _controlled_positional_ids(out).to_numpy(dtype=str)
        id_source = 'posición dentro de condition × age (ID sintético controlado)'
        warnings.warn(
            'El archivo no contiene IDs reales. Se generaron subject_id por posición dentro de '
            'condition × age. La unión entre estímulos supone orden idéntico dentro del estrato.'
        )

    duplicated = out.loc[out['subject_id'].duplicated(keep=False), 'subject_id'].tolist()
    if duplicated:
        raise ValueError(
            'Se esperan estadísticas resumidas con una fila por ratón. '
            f'Se encontraron subject_id duplicados: {duplicated[:10]}'
        )

    out.attrs['subject_id_source'] = id_source
    out.attrs['uses_positional_subject_id'] = id_source.startswith('posición dentro')
    out.attrs['stratum_counts'] = (
        out.groupby(['condition', 'age'], observed=True).size().sort_index().to_dict()
    )
    return out


In [ ]:
# -----------------------------------------------------------------------------
# CARGA DE TABLAS Y DESCUBRIMIENTO DE ESTÍMULOS/COLUMNAS
# -----------------------------------------------------------------------------
META_COLUMNS = {
    'subject_id', 'sample_id', 'mouse_id', 'animal_id', 'raton_id', 'ratón_id',
    'mouse', 'animal', 'subject', 'id',
    'type', 'group', 'class', 'age', 'condition', 'genotype', 'label',
}


def resolve_data_path(base_dir: Path, filename: str) -> Path:
    """Resuelve nombres con o sin .json y sin depender de mayúsculas."""
    base_dir = Path(base_dir)
    requested = Path(filename)
    candidates = [base_dir / requested]
    if requested.suffix.lower() != '.json':
        candidates.append(base_dir / f'{filename}.json')

    for candidate in candidates:
        if candidate.exists():
            return candidate

    if base_dir.exists():
        target_names = {candidate.name.casefold() for candidate in candidates}
        target_stems = {candidate.stem.casefold() for candidate in candidates}
        matches = [
            path for path in base_dir.iterdir()
            if path.is_file()
            and (path.name.casefold() in target_names or path.stem.casefold() in target_stems)
        ]
        if len(matches) == 1:
            return matches[0]
        if len(matches) > 1:
            raise FileExistsError(
                f'Nombre ambiguo {filename!r} en {base_dir}: {[p.name for p in matches]}'
            )

    tried = ', '.join(str(path) for path in candidates)
    raise FileNotFoundError(f'No se encontró {filename!r}. Rutas intentadas: {tried}')


def read_feature_table(
    path: Path,
    *,
    allow_positional_ids: bool = False,
) -> pd.DataFrame:
    if not path.exists():
        raise FileNotFoundError(path)

    df = pd.read_json(path)
    df = normalize_feature_column_names(df)
    result = add_subject_metadata(df, allow_positional_ids=allow_positional_ids)

    if result.attrs.get('subject_id_source') != 'índice original':
        warnings.warn(
            f'{path.name}: el subject_id no provino del índice. '
            f'Fuente utilizada: {result.attrs.get("subject_id_source")}.'
        )
    return result


def _axis_and_stimulus(column: str) -> tuple[str, str] | None:
    s = str(column).strip()
    patterns = [
        r'(?i)^(?:H[_\-\s]*)?(?P<axis>x|y)[_\-\s]*(?P<stim>.*)$',
        r'(?i)^(?P<stim>.*?)[_\-\s]*(?:H[_\-\s]*)?(?P<axis>x|y)$',
    ]
    for pattern in patterns:
        match = re.match(pattern, s)
        if match:
            return match.group('axis').lower(), match.group('stim').strip('_- ')
    return None


def _normalized_name(value: str) -> str:
    return re.sub(r'[^a-z0-9]+', '', str(value).casefold())


def _nonconstant_numeric_columns(df: pd.DataFrame, columns: Sequence[str]) -> list[str]:
    valid: list[str] = []
    for column in columns:
        numeric = pd.to_numeric(df[column], errors='coerce')
        if numeric.notna().sum() and numeric.nunique(dropna=True) > 1:
            valid.append(str(column))
    return valid


def _select_scalar_feature(
    df: pd.DataFrame,
    analysis: str,
    stimulus_id: str,
    path: Path,
) -> str:
    feature_columns = [str(c) for c in df.columns if str(c) not in META_COLUMNS]
    nonconstant = _nonconstant_numeric_columns(df, feature_columns)
    candidates = nonconstant or feature_columns

    preferred_tokens = {
        'energy': ('energylevel3', 'energy3', 'energy'),
        'coactivation': ('coactivation', 'coactiv'),
        'entropy': ('entropy',),
        'kl_divergence': ('kldivergence', 'kldiv', 'kl'),
    }.get(analysis, ())
    matching = [
        column for column in candidates
        if any(token in _normalized_name(column) for token in preferred_tokens)
    ]
    if len(matching) == 1:
        return matching[0]
    if len(candidates) == 1:
        return candidates[0]
    if len(matching) > 1:
        exact = [c for c in matching if _normalized_name(c) in preferred_tokens]
        if len(exact) == 1:
            return exact[0]

    raise ValueError(
        f'No se pudo determinar una única feature para {analysis}, estímulo {stimulus_id}, '
        f'archivo {path.name}. Candidatas numéricas no constantes: {nonconstant}; '
        f'columnas no-meta: {feature_columns}.'
    )


def _select_hurst_axes(df: pd.DataFrame, stimulus_id: str, path: Path) -> dict[str, str]:
    feature_columns = [str(c) for c in df.columns if str(c) not in META_COLUMNS]
    nonconstant = _nonconstant_numeric_columns(df, feature_columns)
    candidates = nonconstant or feature_columns
    axes: dict[str, str] = {}

    for column in candidates:
        normalized = _normalized_name(column)
        parsed = _axis_and_stimulus(column)
        axis = parsed[0] if parsed is not None else None
        if axis is None:
            if normalized in {'hx', 'hurstx', 'xhurst', 'hestimatex', 'hestx'} or normalized.endswith('hx'):
                axis = 'x'
            elif normalized in {'hy', 'hursty', 'yhurst', 'hestimatey', 'hesty'} or normalized.endswith('hy'):
                axis = 'y'
        if axis in {'x', 'y'}:
            if axis in axes:
                raise ValueError(
                    f'Múltiples columnas para el eje {axis} en {path.name}: '
                    f'{axes[axis]!r} y {column!r}.'
                )
            axes[axis] = column

    if set(axes) == {'x', 'y'}:
        return axes
    if len(candidates) == 2:
        warnings.warn(
            f'{path.name}: no se reconocieron H_x/H_y por nombre. Se asignará '
            f'{candidates[0]!r} a x y {candidates[1]!r} a y para H={stimulus_id}.'
        )
        return {'x': candidates[0], 'y': candidates[1]}

    raise ValueError(
        f'No se identificaron H_x y H_y en {path.name}. Columnas candidatas: {candidates}.'
    )


def discover_feature_groups(
    df: pd.DataFrame,
    stimulus_type: str,
    analysis: str,
    expected_stimuli: int,
) -> list[FeatureGroup]:
    override = FEATURE_GROUP_OVERRIDES.get((stimulus_type, analysis))
    if override:
        missing = [c for cols in override.values() for c in cols if c not in df.columns]
        if missing:
            raise KeyError(f'Columnas del override ausentes: {missing}')
        return [FeatureGroup(str(stim), tuple(map(str, cols))) for stim, cols in override.items()]

    feature_columns = [str(c) for c in df.columns if str(c) not in META_COLUMNS]
    if not feature_columns:
        raise ValueError(f'No se encontraron features para {stimulus_type}/{analysis}.')

    if stimulus_type == '2D_stochastic' and analysis == 'hurst':
        parsed = {column: _axis_and_stimulus(column) for column in feature_columns}
        if all(value is not None and value[1] for value in parsed.values()):
            grouped: dict[str, dict[str, str]] = {}
            for column, value in parsed.items():
                axis, stim = value
                grouped.setdefault(stim, {})[axis] = column
            if all(set(axis_map) == {'x', 'y'} for axis_map in grouped.values()):
                groups = [
                    FeatureGroup(str(stim), (axis_map['x'], axis_map['y']))
                    for stim, axis_map in sorted(grouped.items(), key=lambda item: str(item[0]))
                ]
                if len(groups) != expected_stimuli:
                    warnings.warn(
                        f'{stimulus_type}/{analysis}: se esperaban {expected_stimuli} estímulos '
                        f'y se detectaron {len(groups)} pares Hx/Hy.'
                    )
                return groups
        raise ValueError(
            'No fue posible agrupar Hx/Hy. Use un archivo por estímulo o defina '
            'FEATURE_GROUP_OVERRIDES[("2D_stochastic", "hurst")].'
        )

    groups = [FeatureGroup(str(column), (str(column),)) for column in feature_columns]
    if len(groups) != expected_stimuli:
        warnings.warn(
            f'{stimulus_type}/{analysis}: se esperaban {expected_stimuli} estímulos '
            f'y se detectaron {len(groups)} columnas.'
        )
    return groups


def _keep_first_feature_columns(df: pd.DataFrame, n: int) -> pd.DataFrame:
    """Conserva las primeras n features y todas las columnas de metadatos."""
    feature_columns = [str(c) for c in df.columns if str(c) not in META_COLUMNS]
    selected_features = feature_columns[:n]
    if len(feature_columns) < n:
        warnings.warn(f'Se solicitaron {n} features, pero solo existen {len(feature_columns)}.')

    selected_set = set(selected_features) | {str(c) for c in df.columns if str(c) in META_COLUMNS}
    selected_columns = [str(c) for c in df.columns if str(c) in selected_set]
    attrs = df.attrs.copy()
    out = df.loc[:, selected_columns].copy()
    out.attrs.update(attrs)
    return out


def load_analysis_table(
    cfg: StimulusSetConfig,
    analysis: str,
    file_spec: str | Mapping[str, str],
) -> tuple[pd.DataFrame, list[FeatureGroup], list[Path]]:
    """Carga una tabla agregada o combina una tabla independiente por estímulo."""
    if isinstance(file_spec, str):
        path = resolve_data_path(cfg.base_dir, file_spec)
        df = read_feature_table(path)
        if cfg.max_feature_columns is not None:
            df = _keep_first_feature_columns(df, cfg.max_feature_columns)
        groups = discover_feature_groups(
            df, cfg.stimulus_type, analysis, cfg.expected_stimuli,
        )
        return df, groups, [path]

    if not isinstance(file_spec, Mapping) or not file_spec:
        raise TypeError(f'Especificación inválida para {cfg.stimulus_type}/{analysis}.')

    allow_positional = cfg.stimulus_type == '2D_stochastic' and ALLOW_POSITIONAL_IDS_2D
    merged: pd.DataFrame | None = None
    reference_meta: pd.DataFrame | None = None
    reference_id_mode: bool | None = None
    reference_counts: dict | None = None
    groups: list[FeatureGroup] = []
    source_paths: list[Path] = []
    original_counts: dict[str, int] = {}

    for stimulus_id, filename in file_spec.items():
        stimulus_id = str(stimulus_id)
        path = resolve_data_path(cfg.base_dir, str(filename))
        current = read_feature_table(path, allow_positional_ids=allow_positional)
        source_paths.append(path)
        original_counts[stimulus_id] = len(current)

        current_uses_positional = bool(current.attrs.get('uses_positional_subject_id', False))
        current_counts = current.attrs.get('stratum_counts', {})
        current_meta = current[['subject_id', 'age', 'condition']].copy()

        if reference_meta is None:
            reference_meta = current_meta.copy()
            reference_id_mode = current_uses_positional
            reference_counts = current_counts
        else:
            if current_uses_positional != reference_id_mode:
                raise ValueError(
                    f'{cfg.stimulus_type}/{analysis}: algunos archivos contienen IDs reales y '
                    'otros IDs posicionales. Use un esquema consistente.'
                )
            if current_uses_positional and current_counts != reference_counts:
                raise ValueError(
                    f'{cfg.stimulus_type}/{analysis}: los archivos sin IDs reales deben tener los '
                    f'mismos conteos condition × age. Referencia: {reference_counts}; '
                    f'{path.name}: {current_counts}.'
                )

            check = reference_meta.merge(
                current_meta, on='subject_id', how='inner', suffixes=('_ref', '_current'),
                validate='one_to_one',
            )
            inconsistent = check.loc[
                (check['age_ref'] != check['age_current'])
                | (check['condition_ref'] != check['condition_current'])
            ]
            if len(inconsistent):
                raise ValueError(
                    f'Metadatos inconsistentes entre archivos para: '
                    f'{inconsistent["subject_id"].head(10).tolist()}'
                )

        if analysis == 'hurst':
            axes = _select_hurst_axes(current, stimulus_id, path)
            rename_map = {
                axes['x']: f'H_x_{stimulus_id}',
                axes['y']: f'H_y_{stimulus_id}',
            }
            group_columns = (f'H_x_{stimulus_id}', f'H_y_{stimulus_id}')
        else:
            feature = _select_scalar_feature(current, analysis, stimulus_id, path)
            renamed = f'{analysis}_{stimulus_id}'
            rename_map = {feature: renamed}
            group_columns = (renamed,)

        selected = current[['subject_id', *rename_map.keys()]].rename(columns=rename_map)
        if merged is None:
            merged = current_meta.merge(selected, on='subject_id', validate='one_to_one')
        else:
            merged = merged.merge(selected, on='subject_id', how='inner', validate='one_to_one')
        groups.append(FeatureGroup(stimulus_id, group_columns))

    assert merged is not None
    if len(groups) != cfg.expected_stimuli:
        warnings.warn(
            f'{cfg.stimulus_type}/{analysis}: se esperaban {cfg.expected_stimuli} estímulos '
            f'y se configuraron {len(groups)} archivos.'
        )

    minimum_original = min(original_counts.values())
    if len(merged) < minimum_original:
        warnings.warn(
            f'{cfg.stimulus_type}/{analysis}: se usarán {len(merged)} sujetos comunes; '
            f'tamaños originales: {original_counts}.'
        )

    if reference_id_mode:
        warnings.warn(
            f'{cfg.stimulus_type}/{analysis}: se usan IDs posicionales. maxT y las pruebas '
            'globales suponen el mismo orden de ratones dentro de cada estrato.'
        )

    merged.attrs['subject_id_source'] = (
        'unión por ID posicional controlado entre archivos por estímulo'
        if reference_id_mode else 'unión por subject_id real entre archivos por estímulo'
    )
    merged.attrs['uses_positional_subject_id'] = bool(reference_id_mode)
    merged.attrs['source_paths'] = [str(path) for path in source_paths]
    merged.attrs['original_counts'] = original_counts
    merged.attrs['stratum_counts'] = reference_counts or {}
    return merged, groups, source_paths


def _iter_configured_files(
    cfg: StimulusSetConfig,
    analysis: str,
    file_spec: str | Mapping[str, str],
):
    if isinstance(file_spec, str):
        yield 'all', file_spec
    else:
        yield from ((str(stimulus_id), str(filename)) for stimulus_id, filename in file_spec.items())


def validate_project_files(configs: Sequence[StimulusSetConfig]) -> pd.DataFrame:
    """Comprueba cada archivo y la combinación completa de cada análisis."""
    rows: list[dict[str, object]] = []

    for cfg in configs:
        for analysis, file_spec in cfg.files.items():
            for stimulus_id, filename in _iter_configured_files(cfg, analysis, file_spec):
                row: dict[str, object] = {
                    'stimulus_type': cfg.stimulus_type,
                    'analysis': analysis,
                    'stimulus_id': stimulus_id,
                    'configured_name': filename,
                    'path': None,
                    'exists': False,
                    'readable': False,
                    'n_rows': np.nan,
                    'n_subjects': np.nan,
                    'subject_id_source': None,
                    'error': None,
                }
                try:
                    path = resolve_data_path(cfg.base_dir, filename)
                    row['path'] = str(path)
                    row['exists'] = True
                    allow_positional = (
                        cfg.stimulus_type == '2D_stochastic'
                        and isinstance(file_spec, Mapping)
                        and ALLOW_POSITIONAL_IDS_2D
                    )
                    df = read_feature_table(path, allow_positional_ids=allow_positional)
                    row.update({
                        'readable': True,
                        'n_rows': int(len(df)),
                        'n_subjects': int(df['subject_id'].nunique()),
                        'subject_id_source': df.attrs.get('subject_id_source'),
                    })
                except Exception as exc:
                    row['error'] = f'{type(exc).__name__}: {exc}'
                rows.append(row)

            combined_row: dict[str, object] = {
                'stimulus_type': cfg.stimulus_type,
                'analysis': analysis,
                'stimulus_id': '__combined__',
                'configured_name': '',
                'path': '',
                'exists': True,
                'readable': False,
                'n_rows': np.nan,
                'n_subjects': np.nan,
                'subject_id_source': None,
                'error': None,
            }
            try:
                combined, groups, paths = load_analysis_table(cfg, analysis, file_spec)
                combined_row.update({
                    'path': '; '.join(str(path) for path in paths),
                    'readable': True,
                    'n_rows': int(len(combined)),
                    'n_subjects': int(combined['subject_id'].nunique()),
                    'subject_id_source': combined.attrs.get('subject_id_source'),
                    'configured_name': f'{len(groups)} estímulos combinados',
                })
            except Exception as exc:
                combined_row['error'] = f'{type(exc).__name__}: {exc}'
            rows.append(combined_row)

    result = pd.DataFrame(rows)
    problems = result.loc[~result['exists'] | ~result['readable']]
    if problems.empty:
        print(f'Se validaron correctamente {len(result)} entradas.')
    else:
        print(f'Se detectaron problemas en {len(problems)} de {len(result)} entradas.')
    return result


In [ ]:
for cfg in STIMULUS_SETS:
    for analysis, file_spec in cfg.files.items():
        try:
            df, _, _ = load_analysis_table(cfg, analysis, file_spec)
            print(f'\n{cfg.stimulus_type} | {analysis}')
            print(df.head())
        except FileNotFoundError:
            pass


## 2. Clasificador y validación cruzada

El preprocesamiento se ejecuta **dentro** de cada fold mediante un `Pipeline` (imputación, estandarización y SVM). Esto evita fuga de información.

El modo recomendado es `MODEL_POLICY='fixed'`: los hiperparámetros se fijan antes de mirar los resultados. El modo `saved_template` carga un modelo, extrae el estimador y lo clona; esto evita reutilizar el ajuste, pero **no corrige** el sesgo si sus hiperparámetros fueron elegidos con todo el mismo conjunto de datos. En ese caso, la selección de hiperparámetros debería repetirse dentro de cada fold y de cada permutación (validación anidada).


In [ ]:
# -----------------------------------------------------------------------------
# MODELO, VALIDACIÓN CRUZADA Y PUNTAJE
# -----------------------------------------------------------------------------
ANALYSIS_ALIASES = {
    'energy': ['energy'],
    'coactivation': ['coactivation', 'coactiv'],
    'entropy': ['entropy'],
    'kl_divergence': ['kl_divergence', 'kl_div', 'kldiv'],
    'hurst': ['hurst', 'h'],
}
CONTRAST_MODEL_LABEL = {
    'age_main': 'age',
    'condition_main': 'cond',
    'condition_at_3m': 'cond',
    'condition_at_6m': 'cond',
    'age_within_WT': 'age',
    'age_within_5xFAD': 'age',
}


def fixed_svm() -> Pipeline:
    return Pipeline([
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
        ('svm', SVC(
            kernel=SVM_KERNEL,
            C=SVM_C,
            class_weight=SVM_CLASS_WEIGHT,
        )),
    ])


def _extract_estimator(loaded):
    if isinstance(loaded, dict):
        for key in ('model', 'estimator', 'classifier', 'svm'):
            if key in loaded:
                return loaded[key]
        raise KeyError(f'El archivo joblib contiene un dict sin estimator conocido: {list(loaded)}')
    return loaded


def find_saved_model(
    cfg: StimulusSetConfig,
    analysis: str,
    contrast: str,
    stimulus_id: str,
) -> Path | None:
    override = SAVED_MODEL_OVERRIDES.get((cfg.stimulus_type, analysis, contrast, stimulus_id))
    if override:
        path = PROJECT_ROOT / override
        if not path.exists():
            raise FileNotFoundError(path)
        return path

    if not cfg.model_dir.exists():
        return None

    label = CONTRAST_MODEL_LABEL.get(contrast, contrast)

    stimulus_tokens = [str(stimulus_id)]
    decimal_match = re.fullmatch(r'0[._-]?([0-9]+)', str(stimulus_id))
    if decimal_match:
        digits = decimal_match.group(1)
        stimulus_tokens.extend([f'0.{digits}', f'0_{digits}', f'0{digits}', f'h{digits}', digits])
    stimulus_tokens = list(dict.fromkeys(stimulus_tokens))

    candidates: list[Path] = []
    for alias in ANALYSIS_ALIASES.get(analysis, [analysis]):
        bases = [f'{alias}_{label}_svm', f'{alias}_{label}']
        for token in stimulus_tokens:
            bases.extend([
                f'{alias}_{label}_svm_{token}',
                f'{alias}_{label}_{token}',
            ])
        for base in bases:
            for suffix in ('.pkl', '.joblib'):
                path = cfg.model_dir / f'{base}{suffix}'
                if path.exists():
                    candidates.append(path)
    unique = list(dict.fromkeys(candidates))
    if len(unique) > 1:
        warnings.warn(f'Múltiples modelos candidatos; se utilizará {unique[0]}: {unique}')
    return unique[0] if unique else None


def make_estimator(
    cfg: StimulusSetConfig,
    analysis: str,
    contrast: str,
    stimulus_id: str,
):
    if MODEL_POLICY == 'fixed':
        return fixed_svm(), 'fixed_svm'
    if MODEL_POLICY == 'saved_template':
        path = find_saved_model(cfg, analysis, contrast, stimulus_id)
        if path is None:
            warnings.warn(
                f'No se encontró modelo para {cfg.stimulus_type}/{analysis}/{contrast}/{stimulus_id}; '
                'se usará el SVM fijo.'
            )
            return fixed_svm(), 'fixed_svm_fallback'
        estimator = _extract_estimator(joblib.load(path))
        return clone(estimator), str(path)
    raise ValueError(f'MODEL_POLICY no reconocido: {MODEL_POLICY}')


def make_cv_splits(y: np.ndarray, seed: int) -> list[tuple[np.ndarray, np.ndarray]]:
    y = np.asarray(y)
    if CV_SCHEME == 'loo':
        return list(LeaveOneOut().split(np.zeros((len(y), 1)), y))
    if CV_SCHEME == 'stratified_kfold':
        counts = pd.Series(y).value_counts()
        n_splits = min(N_SPLITS, int(counts.min()))
        if n_splits < 2:
            raise ValueError(f'No hay suficientes individuos por clase: {counts.to_dict()}')
        cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=seed)
        return list(cv.split(np.zeros((len(y), 1)), y))
    raise ValueError(f'CV_SCHEME no reconocido: {CV_SCHEME}')


def cv_scores(estimator, X: np.ndarray, y: np.ndarray, seed: int) -> dict[str, float]:
    splits = make_cv_splits(y, seed)
    predictions = cross_val_predict(
        clone(estimator), X, y, cv=splits, method='predict', n_jobs=1,
    )
    return {
        'balanced_accuracy': balanced_accuracy_score(y, predictions),
        'accuracy': accuracy_score(y, predictions),
    }


def binomial_accuracy_threshold(n: int, n_classes: int = 2) -> float:
    """Umbral diagnóstico calculado exclusivamente con ALPHA = 0.05."""
    chance = 1 / n_classes
    # Convención de la tabla del paper: cuantil binomial 1-ALPHA.
    k = int(binom.ppf(1 - ALPHA, n, chance))
    return min(k / n, 1.0)


In [ ]:
# -----------------------------------------------------------------------------
# TAMAÑOS DE EFECTO
# -----------------------------------------------------------------------------
def cohen_d(x_negative: np.ndarray, x_positive: np.ndarray) -> float:
    x0 = np.asarray(x_negative, dtype=float)
    x1 = np.asarray(x_positive, dtype=float)
    n0, n1 = len(x0), len(x1)
    if n0 < 2 or n1 < 2:
        return np.nan
    pooled_var = ((n0 - 1) * np.var(x0, ddof=1) + (n1 - 1) * np.var(x1, ddof=1)) / (n0 + n1 - 2)
    if pooled_var <= 0:
        return np.nan
    return (np.mean(x1) - np.mean(x0)) / np.sqrt(pooled_var)


def hedges_g_from_d(d: float, n0: int, n1: int) -> float:
    if not np.isfinite(d):
        return np.nan
    df = n0 + n1 - 2
    if df <= 1:
        return np.nan
    correction = 1 - 3 / (4 * df - 1)
    return correction * d


def cliffs_delta(x_negative: np.ndarray, x_positive: np.ndarray) -> float:
    """Positivo cuando la clase positiva tiende a valores mayores."""
    x0 = np.asarray(x_negative, dtype=float)
    x1 = np.asarray(x_positive, dtype=float)
    comparisons = np.sign(x1[:, None] - x0[None, :])
    return float(comparisons.mean())


def regularized_mahalanobis(X_negative: np.ndarray, X_positive: np.ndarray) -> float:
    X0 = np.asarray(X_negative, dtype=float)
    X1 = np.asarray(X_positive, dtype=float)
    if X0.ndim == 1:
        X0 = X0[:, None]
    if X1.ndim == 1:
        X1 = X1[:, None]
    if len(X0) < 2 or len(X1) < 2:
        return np.nan
    diff = X1.mean(axis=0) - X0.mean(axis=0)
    residuals = np.vstack([X0 - X0.mean(axis=0), X1 - X1.mean(axis=0)])
    covariance = LedoitWolf().fit(residuals).covariance_
    value = float(diff @ np.linalg.pinv(covariance) @ diff)
    return float(np.sqrt(max(value, 0.0)))


def bootstrap_effect_ci(
    x_negative: np.ndarray,
    x_positive: np.ndarray,
    statistic: Callable[[np.ndarray, np.ndarray], float],
    n_bootstrap: int,
    seed: int,
) -> tuple[float, float]:
    rng = np.random.default_rng(seed)
    x0 = np.asarray(x_negative, dtype=float)
    x1 = np.asarray(x_positive, dtype=float)
    values = np.empty(n_bootstrap, dtype=float)
    for i in range(n_bootstrap):
        b0 = rng.choice(x0, size=len(x0), replace=True)
        b1 = rng.choice(x1, size=len(x1), replace=True)
        values[i] = statistic(b0, b1)
    values = values[np.isfinite(values)]
    if not len(values):
        return np.nan, np.nan
    return tuple(np.quantile(values, [0.025, 0.975]))


def effect_size_rows(
    frame: pd.DataFrame,
    feature_group: FeatureGroup,
    contrast: Contrast,
    context: Mapping[str, object],
    seed: int,
) -> list[dict]:
    rows = []
    negative_mask = frame[contrast.target] == contrast.negative_class
    positive_mask = frame[contrast.target] == contrast.positive_class

    for feature_column in feature_group.columns:
        x0 = frame.loc[negative_mask, feature_column].to_numpy(dtype=float)
        x1 = frame.loc[positive_mask, feature_column].to_numpy(dtype=float)
        d = cohen_d(x0, x1)
        g = hedges_g_from_d(d, len(x0), len(x1))
        d_low, d_high = bootstrap_effect_ci(x0, x1, cohen_d, N_BOOTSTRAP_EFFECT, seed)

        def g_stat(a, b):
            return hedges_g_from_d(cohen_d(a, b), len(a), len(b))

        g_low, g_high = bootstrap_effect_ci(x0, x1, g_stat, N_BOOTSTRAP_EFFECT, seed + 1)
        rows.append({
            **context,
            'feature_column': feature_column,
            'negative_class': contrast.negative_class,
            'positive_class': contrast.positive_class,
            'n_negative': len(x0),
            'n_positive': len(x1),
            'mean_negative': float(np.mean(x0)),
            'mean_positive': float(np.mean(x1)),
            'sd_negative': float(np.std(x0, ddof=1)) if len(x0) > 1 else np.nan,
            'sd_positive': float(np.std(x1, ddof=1)) if len(x1) > 1 else np.nan,
            'cohen_d': d,
            'cohen_d_ci_low': d_low,
            'cohen_d_ci_high': d_high,
            'hedges_g': g,
            'hedges_g_ci_low': g_low,
            'hedges_g_ci_high': g_high,
            'cliffs_delta': cliffs_delta(x0, x1),
        })
    return rows


## 3. Prueba de permutación sincronizada y maxT

Para cada combinación `tipo de estímulo × metodología × contraste`:

1. Se utiliza el mismo conjunto completo de ratones para todos sus estímulos.
2. Se calcula la exactitud balanceada observada con todo el pipeline de CV.
3. En cada permutación se intercambia la etiqueta objetivo dentro de los bloques del factor de control.
4. Se vuelve a entrenar y evaluar el SVM para cada estímulo.
5. El p-valor usa la corrección de Monte Carlo `(b + 1)/(B + 1)`, por lo que nunca es cero.
6. La distribución del máximo entre estímulos entrega `p_maxT_fwer`, que controla el error familiar al escoger el mejor estímulo.
7. Se calculan dos pruebas globales por metodología:
   - `global_any_p`: evidencia de que al menos un estímulo separa (máximo).
   - `global_consistency_p`: evidencia de desempeño promedio entre estímulos (media).


In [ ]:
# -----------------------------------------------------------------------------
# PERMUTACIONES RESTRINGIDAS Y ANÁLISIS DE UNA FAMILIA DE ESTÍMULOS
# -----------------------------------------------------------------------------
def restricted_permutation(
    y: np.ndarray,
    blocks: np.ndarray | None,
    rng: np.random.Generator,
) -> np.ndarray:
    y = np.asarray(y)
    if blocks is None:
        return rng.permutation(y)
    permuted = y.copy()
    blocks = np.asarray(blocks)
    for block in pd.unique(blocks):
        positions = np.flatnonzero(blocks == block)
        permuted[positions] = rng.permutation(y[positions])
    return permuted


def conservative_quantile(values: np.ndarray, q: float) -> float:
    try:
        return float(np.quantile(values, q, method='higher'))
    except TypeError:
        return float(np.quantile(values, q, interpolation='higher'))


def monte_carlo_p(null_values: np.ndarray, observed: float) -> float:
    null_values = np.asarray(null_values, dtype=float)
    valid = null_values[np.isfinite(null_values)]
    return (1 + int(np.sum(valid >= observed))) / (len(valid) + 1)


def _one_permutation_scores(
    seed: int,
    y: np.ndarray,
    blocks: np.ndarray | None,
    X_list: Sequence[np.ndarray],
    estimators: Sequence,
) -> np.ndarray:
    rng = np.random.default_rng(seed)
    y_perm = restricted_permutation(y, blocks, rng)
    scores = np.empty(len(X_list), dtype=float)
    for j, (X, estimator) in enumerate(zip(X_list, estimators)):
        scores[j] = cv_scores(estimator, X, y_perm, RANDOM_STATE)['balanced_accuracy']
    return scores


def _permutation_batch_scores(
    seeds: Sequence[int],
    y: np.ndarray,
    blocks: np.ndarray | None,
    X_list: Sequence[np.ndarray],
    estimators: Sequence,
) -> np.ndarray:
    """Ejecuta un lote de permutaciones dentro de un único trabajo joblib."""
    return np.vstack([
        _one_permutation_scores(seed, y, blocks, X_list, estimators)
        for seed in seeds
    ])


def analyze_method_family(
    cfg: StimulusSetConfig,
    analysis: str,
    file_spec: str | Mapping[str, str],
    contrast: Contrast,
) -> tuple[pd.DataFrame, pd.DataFrame, dict, dict, list[dict]]:
    start = time.perf_counter()
    df, groups, source_paths = load_analysis_table(cfg, analysis, file_spec)
    contrasted = apply_contrast(df, contrast)

    all_feature_columns = sorted({column for group in groups for column in group.columns})
    numeric = contrasted[all_feature_columns].apply(pd.to_numeric, errors='coerce')
    complete_mask = numeric.notna().all(axis=1)
    frame = pd.concat([
        contrasted.loc[complete_mask, ['subject_id', 'age', 'condition']].reset_index(drop=True),
        numeric.loc[complete_mask].reset_index(drop=True),
    ], axis=1)

    if len(frame) < 4:
        raise ValueError(f'Muy pocos casos completos para {cfg.stimulus_type}/{analysis}/{contrast.name}.')

    class_counts = frame[contrast.target].value_counts()
    if set(class_counts.index) != {contrast.negative_class, contrast.positive_class}:
        raise ValueError(
            f'Clases incompletas en {cfg.stimulus_type}/{analysis}/{contrast.name}: '
            f'{class_counts.to_dict()}'
        )
    if int(class_counts.min()) < 2:
        raise ValueError(f'Se requieren al menos 2 individuos por clase: {class_counts.to_dict()}')

    y = frame[contrast.target].to_numpy()
    blocks = frame[contrast.nuisance].to_numpy() if contrast.nuisance else None

    if contrast.nuisance:
        contingency = pd.crosstab(frame[contrast.nuisance], frame[contrast.target])
        missing_within_block = [
            block for block, row in contingency.iterrows()
            if any(row.get(label, 0) == 0 for label in (contrast.negative_class, contrast.positive_class))
        ]
        if missing_within_block:
            raise ValueError(
                f'No es posible permutar {contrast.target} dentro de {contrast.nuisance}: '
                f'los bloques {missing_within_block} no contienen ambas clases. '
                f'Tabla: {contingency.to_dict()}'
            )

    X_list: list[np.ndarray] = []
    estimators = []
    estimator_sources = []
    for group in groups:
        X_list.append(frame[list(group.columns)].to_numpy(dtype=float))
        estimator, source = make_estimator(cfg, analysis, contrast.name, group.stimulus_id)
        estimators.append(estimator)
        estimator_sources.append(source)

    print(
        f'  Sujetos completos: {len(frame)} | estímulos: {len(groups)} | '
        f'permutaciones: {N_PERMUTATIONS}'
    )

    observed_balanced = np.empty(len(groups), dtype=float)
    observed_accuracy = np.empty(len(groups), dtype=float)
    for j, (X, estimator) in enumerate(zip(X_list, estimators)):
        scores = cv_scores(estimator, X, y, RANDOM_STATE)
        observed_balanced[j] = scores['balanced_accuracy']
        observed_accuracy[j] = scores['accuracy']

    seed_sequence = np.random.SeedSequence(RANDOM_STATE).spawn(N_PERMUTATIONS)
    permutation_seeds = [int(sequence.generate_state(1)[0]) for sequence in seed_sequence]
    seed_batches = [
        permutation_seeds[start_idx:start_idx + PERMUTATION_BATCH_SIZE]
        for start_idx in range(0, len(permutation_seeds), PERMUTATION_BATCH_SIZE)
    ]

    permutation_batches = Parallel(
        n_jobs=N_JOBS,
        verbose=JOBLIB_VERBOSE,
        prefer='processes',
        batch_size=1,
    )(
        delayed(_permutation_batch_scores)(batch, y, blocks, X_list, estimators)
        for batch in seed_batches
    )
    null_matrix = np.vstack(permutation_batches)
    max_null = np.nanmax(null_matrix, axis=1)
    mean_null = np.nanmean(null_matrix, axis=1)

    max_threshold = conservative_quantile(max_null, 1 - ALPHA)
    raw_rows = []
    effect_rows_all: list[dict] = []
    for j, group in enumerate(groups):
        raw_p = monte_carlo_p(null_matrix[:, j], observed_balanced[j])
        max_t_p = monte_carlo_p(max_null, observed_balanced[j])
        context = {
            'stimulus_type': cfg.stimulus_type,
            'analysis': analysis,
            'contrast': contrast.name,
            'stimulus_id': group.stimulus_id,
        }
        effect_rows = effect_size_rows(
            frame, group, contrast, context,
            seed=RANDOM_STATE + j * 17,
        )
        effect_rows_all.extend(effect_rows)
        hedges_values = np.array([row['hedges_g'] for row in effect_rows], dtype=float)

        negative = frame[contrast.target] == contrast.negative_class
        positive = frame[contrast.target] == contrast.positive_class
        mahalanobis = regularized_mahalanobis(
            frame.loc[negative, list(group.columns)].to_numpy(dtype=float),
            frame.loc[positive, list(group.columns)].to_numpy(dtype=float),
        )
        raw_rows.append({
            **context,
            'feature_columns': ', '.join(group.columns),
            'n_features': len(group.columns),
            'n_total': len(frame),
            'n_negative': int((y == contrast.negative_class).sum()),
            'n_positive': int((y == contrast.positive_class).sum()),
            'negative_class': contrast.negative_class,
            'positive_class': contrast.positive_class,
            'nuisance_factor': contrast.nuisance or '',
            'cv_scheme': CV_SCHEME,
            'model_source': estimator_sources[j],
            'balanced_accuracy': observed_balanced[j],
            'accuracy': observed_accuracy[j],
            'theoretical_chance': 0.5,
            'binomial_threshold_diagnostic': binomial_accuracy_threshold(len(frame), 2),
            'permutation_threshold_raw': conservative_quantile(null_matrix[:, j], 1 - ALPHA),
            'permutation_threshold_maxT': max_threshold,
            'p_raw': raw_p,
            'p_maxT_fwer_within_analysis': max_t_p,
            'significant_raw': raw_p <= ALPHA,
            'significant_maxT': max_t_p <= ALPHA,
            'max_abs_hedges_g': float(np.nanmax(np.abs(hedges_values))),
            'mean_abs_hedges_g': float(np.nanmean(np.abs(hedges_values))),
            'mahalanobis_distance': mahalanobis,
        })

    tests_df = pd.DataFrame(raw_rows)
    effects_df = pd.DataFrame(effect_rows_all)
    elapsed = time.perf_counter() - start
    summary = {
        'stimulus_type': cfg.stimulus_type,
        'analysis': analysis,
        'contrast': contrast.name,
        'cv_scheme': CV_SCHEME,
        'n_total': len(frame),
        'n_stimuli': len(groups),
        'best_stimulus': groups[int(np.argmax(observed_balanced))].stimulus_id,
        'best_balanced_accuracy': float(np.max(observed_balanced)),
        'mean_balanced_accuracy': float(np.mean(observed_balanced)),
        'global_any_statistic_max': float(np.max(observed_balanced)),
        'global_any_p': monte_carlo_p(max_null, float(np.max(observed_balanced))),
        'global_consistency_statistic_mean': float(np.mean(observed_balanced)),
        'global_consistency_p': monte_carlo_p(mean_null, float(np.mean(observed_balanced))),
        'elapsed_seconds': elapsed,
    }
    quality = {
        'stimulus_type': cfg.stimulus_type,
        'analysis': analysis,
        'contrast': contrast.name,
        'source_file': '; '.join(str(path) for path in source_paths),
        'n_original': len(contrasted),
        'n_complete_common': len(frame),
        'n_removed_missing': int((~complete_mask).sum()),
        'class_counts': json.dumps(class_counts.to_dict(), ensure_ascii=False),
        'n_stimuli_detected': len(groups),
        'expected_stimuli': cfg.expected_stimuli,
        'subject_id_source': df.attrs.get('subject_id_source', ''),
        'uses_positional_subject_id': bool(df.attrs.get('uses_positional_subject_id', False)),
        'positional_id_caveat': (
            'El orden dentro de condition × age debe representar a los mismos ratones en todos '
            'los archivos.' if df.attrs.get('uses_positional_subject_id', False) else ''
        ),
    }
    null_payload = {
        'key': f'{cfg.stimulus_type}__{analysis}__{contrast.name}',
        'null_matrix': null_matrix,
        'stimulus_ids': np.array([g.stimulus_id for g in groups], dtype=object),
    }
    print(f'  Finalizado en {elapsed / 60:.2f} min.')
    return tests_df, effects_df, summary, null_payload, [quality]


In [ ]:
# -----------------------------------------------------------------------------
# CHECKPOINTS, CORRECCIONES MÚLTIPLES Y RANKING
# -----------------------------------------------------------------------------
def apply_multiple_testing(results: pd.DataFrame) -> pd.DataFrame:
    out = results.copy()
    out['p_fdr_bh_primary_family'] = np.nan
    out['sig_fdr_bh_primary_family'] = False
    out['p_fdr_by_primary_family'] = np.nan
    out['sig_fdr_by_primary_family'] = False

    family_columns = ['stimulus_type', 'contrast', 'cv_scheme']
    for _, index in out.groupby(family_columns, sort=False).groups.items():
        index = list(index)
        p = out.loc[index, 'p_raw'].to_numpy(dtype=float)
        reject_bh, p_bh, _, _ = multipletests(p, alpha=ALPHA, method='fdr_bh')
        reject_by, p_by, _, _ = multipletests(p, alpha=ALPHA, method='fdr_by')
        out.loc[index, 'p_fdr_bh_primary_family'] = p_bh
        out.loc[index, 'sig_fdr_bh_primary_family'] = reject_bh
        out.loc[index, 'p_fdr_by_primary_family'] = p_by
        out.loc[index, 'sig_fdr_by_primary_family'] = reject_by

    out['p_fdr_bh_within_analysis'] = np.nan
    out['sig_fdr_bh_within_analysis'] = False
    within_columns = ['stimulus_type', 'contrast', 'analysis', 'cv_scheme']
    for _, index in out.groupby(within_columns, sort=False).groups.items():
        index = list(index)
        p = out.loc[index, 'p_raw'].to_numpy(dtype=float)
        reject, p_adj, _, _ = multipletests(p, alpha=ALPHA, method='fdr_bh')
        out.loc[index, 'p_fdr_bh_within_analysis'] = p_adj
        out.loc[index, 'sig_fdr_bh_within_analysis'] = reject
    return out


def _safe_name(value: str) -> str:
    return re.sub(r'[^A-Za-z0-9_.-]+', '_', value)


def current_run_configuration() -> dict[str, object]:
    return {
        'checkpoint_version': CHECKPOINT_VERSION,
        'run_profile': RUN_PROFILE,
        'n_permutations': N_PERMUTATIONS,
        'n_bootstrap_effect': N_BOOTSTRAP_EFFECT,
        'random_state': RANDOM_STATE,
        'alpha': ALPHA,
        'cv_scheme': CV_SCHEME,
        'n_splits': N_SPLITS,
        'model_policy': MODEL_POLICY,
        'svm_kernel': SVM_KERNEL,
        'svm_c': SVM_C,
        'svm_class_weight': SVM_CLASS_WEIGHT,
        'permutation_batch_size': PERMUTATION_BATCH_SIZE,
        'allow_positional_ids_2d': ALLOW_POSITIONAL_IDS_2D,
    }


RUN_TAG = _safe_name(
    f'v{CHECKPOINT_VERSION}_{RUN_PROFILE}_{CV_SCHEME}_perm{N_PERMUTATIONS}_'
    f'boot{N_BOOTSTRAP_EFFECT}_{MODEL_POLICY}_{SVM_KERNEL}_C{SVM_C}'
)
PARTIAL_DIR = OUTPUT_DIR / 'partial_results' / RUN_TAG
PARTIAL_DIR.mkdir(parents=True, exist_ok=True)


def get_stimulus_config(stimulus_type: str) -> StimulusSetConfig:
    matches = [cfg for cfg in STIMULUS_SETS if cfg.stimulus_type == stimulus_type]
    if len(matches) != 1:
        raise ValueError(f'Configuración no única para {stimulus_type!r}: {len(matches)} coincidencias.')
    return matches[0]


def get_contrast(contrast_name: str) -> Contrast:
    matches = [contrast for contrast in CONTRASTS if contrast.name == contrast_name]
    if len(matches) != 1:
        raise ValueError(f'Contraste no único para {contrast_name!r}: {len(matches)} coincidencias.')
    return matches[0]


def build_analysis_tasks() -> list[tuple[str, str, str]]:
    tasks: list[tuple[str, str, str]] = []
    for cfg in STIMULUS_SETS:
        if SELECTED_STIMULUS_TYPES is not None and cfg.stimulus_type not in SELECTED_STIMULUS_TYPES:
            continue
        for analysis in cfg.files:
            if SELECTED_ANALYSES is not None and analysis not in SELECTED_ANALYSES:
                continue
            for contrast in CONTRASTS:
                if SELECTED_CONTRASTS is not None and contrast.name not in SELECTED_CONTRASTS:
                    continue
                tasks.append((cfg.stimulus_type, analysis, contrast.name))
    return tasks


def _source_signature(
    cfg: StimulusSetConfig,
    analysis: str,
) -> list[dict[str, object]]:
    file_spec = cfg.files[analysis]
    signature: list[dict[str, object]] = []
    for stimulus_id, filename in _iter_configured_files(cfg, analysis, file_spec):
        path = resolve_data_path(cfg.base_dir, filename)
        stat = path.stat()
        signature.append({
            'stimulus_id': stimulus_id,
            'path': str(path.resolve()),
            'size': stat.st_size,
            'mtime_ns': stat.st_mtime_ns,
        })
    return signature


def _checkpoint_path(stimulus_type: str, analysis: str, contrast_name: str) -> Path:
    filename = _safe_name(f'{stimulus_type}__{analysis}__{contrast_name}') + '.joblib'
    return PARTIAL_DIR / filename


def _expected_task_signature(
    stimulus_type: str,
    analysis: str,
    contrast_name: str,
) -> dict[str, object]:
    cfg = get_stimulus_config(stimulus_type)
    return {
        'task': (stimulus_type, analysis, contrast_name),
        'configuration': current_run_configuration(),
        'sources': _source_signature(cfg, analysis),
    }


def run_analysis_task(
    stimulus_type: str,
    analysis: str,
    contrast_name: str,
    *,
    overwrite: bool = False,
) -> dict:
    """Ejecuta y guarda una unidad independiente: tipo × análisis × contraste."""
    cfg = get_stimulus_config(stimulus_type)
    contrast = get_contrast(contrast_name)
    if analysis not in cfg.files:
        raise KeyError(f'{analysis!r} no está configurado para {stimulus_type!r}.')

    checkpoint_path = _checkpoint_path(stimulus_type, analysis, contrast_name)
    expected_signature = _expected_task_signature(stimulus_type, analysis, contrast_name)

    if checkpoint_path.exists() and not overwrite:
        try:
            payload = joblib.load(checkpoint_path)
            if payload.get('task_signature') == expected_signature:
                print(f'Checkpoint compatible: {checkpoint_path.name}')
                return payload
            print(f'Checkpoint desactualizado; se recalculará: {checkpoint_path.name}')
        except Exception as exc:
            print(f'Checkpoint ilegible; se recalculará ({type(exc).__name__}: {exc}).')

    print('\n' + '=' * 90)
    print(f'Analizando {stimulus_type} | {analysis} | {contrast_name}')
    print('=' * 90)

    tests, effects, summary, null_payload, quality = analyze_method_family(
        cfg, analysis, cfg.files[analysis], contrast,
    )
    payload = {
        'task_signature': expected_signature,
        'tests_raw': tests,
        'effects': effects,
        'method_summary_raw': pd.DataFrame([summary]),
        'quality': pd.DataFrame(quality),
        'null_payload': null_payload,
    }
    joblib.dump(payload, checkpoint_path, compress=CHECKPOINT_COMPRESSION)
    print(f'Checkpoint guardado: {checkpoint_path}')
    return payload


def checkpoint_status(
    tasks: Sequence[tuple[str, str, str]] | None = None,
) -> pd.DataFrame:
    tasks = list(tasks or build_analysis_tasks())
    rows = []
    for stimulus_type, analysis, contrast_name in tasks:
        path = _checkpoint_path(stimulus_type, analysis, contrast_name)
        compatible = False
        error = None
        if path.exists():
            try:
                payload = joblib.load(path)
                compatible = payload.get('task_signature') == _expected_task_signature(
                    stimulus_type, analysis, contrast_name,
                )
            except Exception as exc:
                error = f'{type(exc).__name__}: {exc}'
        rows.append({
            'stimulus_type': stimulus_type,
            'analysis': analysis,
            'contrast': contrast_name,
            'checkpoint_exists': path.exists(),
            'compatible': compatible,
            'ready': path.exists() and compatible,
            'path': str(path),
            'error': error,
        })
    return pd.DataFrame(rows)


def run_pending_tasks(
    tasks: Sequence[tuple[str, str, str]] | None = None,
    *,
    overwrite: bool = False,
    stop_on_error: bool = True,
) -> list[dict]:
    tasks = list(tasks or build_analysis_tasks())
    outputs = []
    total = len(tasks)
    for index, (stimulus_type, analysis, contrast_name) in enumerate(tasks, start=1):
        print(f'\nTarea {index}/{total}')
        try:
            outputs.append(run_analysis_task(
                stimulus_type, analysis, contrast_name, overwrite=overwrite,
            ))
        except Exception as exc:
            print(f'ERROR: {stimulus_type}/{analysis}/{contrast_name}: {type(exc).__name__}: {exc}')
            if stop_on_error:
                raise
    return outputs


def combine_partial_results(
    tasks: Sequence[tuple[str, str, str]] | None = None,
):
    tasks = list(tasks or build_analysis_tasks())
    status = checkpoint_status(tasks)
    missing = status.loc[~status['ready']]
    if len(missing):
        raise FileNotFoundError(
            'Faltan checkpoints compatibles. Ejecute las tareas pendientes antes de combinar:\n'
            + missing[['stimulus_type', 'analysis', 'contrast']].to_string(index=False)
        )

    partials = [
        joblib.load(_checkpoint_path(stimulus_type, analysis, contrast_name))
        for stimulus_type, analysis, contrast_name in tasks
    ]

    tests_raw = pd.concat([p['tests_raw'] for p in partials], ignore_index=True)
    results = apply_multiple_testing(tests_raw)
    effects = pd.concat([p['effects'] for p in partials], ignore_index=True)
    quality = pd.concat([p['quality'] for p in partials], ignore_index=True)
    method_summary = pd.concat([p['method_summary_raw'] for p in partials], ignore_index=True)

    for p_column in ['global_any_p', 'global_consistency_p']:
        adjusted_column = f'{p_column}_fdr_bh'
        significant_column = f'{p_column}_sig_fdr_bh'
        method_summary[adjusted_column] = np.nan
        method_summary[significant_column] = False
        for _, index in method_summary.groupby(
            ['stimulus_type', 'contrast', 'cv_scheme'], sort=False,
        ).groups.items():
            index = list(index)
            reject, adjusted, _, _ = multipletests(
                method_summary.loc[index, p_column].to_numpy(dtype=float),
                alpha=ALPHA,
                method='fdr_bh',
            )
            method_summary.loc[index, adjusted_column] = adjusted
            method_summary.loc[index, significant_column] = reject

    ranking = results.sort_values(
        [
            'stimulus_type', 'contrast',
            'p_maxT_fwer_within_analysis',
            'p_fdr_bh_primary_family',
            'balanced_accuracy',
            'max_abs_hedges_g',
        ],
        ascending=[True, True, True, True, False, False],
    ).reset_index(drop=True)
    ranking['rank_within_question'] = (
        ranking.groupby(['stimulus_type', 'contrast']).cumcount() + 1
    )

    null_payloads = [p['null_payload'] for p in partials]
    return results, effects, method_summary, quality, ranking, null_payloads


def run_all_analyses(*, overwrite: bool = False):
    """Atajo: ejecuta checkpoints pendientes y combina todos los resultados."""
    tasks = build_analysis_tasks()
    run_pending_tasks(tasks, overwrite=overwrite)
    return combine_partial_results(tasks)


In [ ]:
# -----------------------------------------------------------------------------
# EXPORTACIÓN A EXCEL
# -----------------------------------------------------------------------------
def _excel_safe_frame(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    for column in out.columns:
        if out[column].map(lambda x: isinstance(x, (dict, list, tuple, np.ndarray))).any():
            out[column] = out[column].map(
                lambda x: json.dumps(x, ensure_ascii=False, default=str)
                if isinstance(x, (dict, list, tuple, np.ndarray)) else x
            )
    return out


def export_excel(
    path: Path,
    results: pd.DataFrame,
    effects: pd.DataFrame,
    method_summary: pd.DataFrame,
    quality: pd.DataFrame,
    ranking: pd.DataFrame,
) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    notes = pd.DataFrame({
        'Campo': [
            'Nivel de significancia',
            'p_raw',
            'p_maxT_fwer_within_analysis',
            'p_fdr_bh_primary_family',
            'p_fdr_by_primary_family',
            'global_any_p',
            'global_consistency_p',
            'Cohen d / Hedges g',
            'Signo del efecto',
            'IDs posicionales 2D',
        ],
        'Interpretación': [
            'Todo el notebook utiliza exclusivamente alpha = 0.05 para pruebas individuales, maxT, FWER y FDR.',
            'p de permutación para un estímulo y metodología concretos.',
            'FWER maxT al seleccionar entre estímulos de la misma metodología.',
            'FDR BH entre todas las metodologías y estímulos de una pregunta biológica.',
            'FDR BY conservador ante dependencia arbitraria.',
            'Prueba global: existe al menos un estímulo separador para la metodología.',
            'Prueba global: la metodología muestra separación promedio entre estímulos.',
            'Tamaño de efecto descriptivo; Hedges g corrige el sesgo de Cohen d con n pequeño.',
            'Positivo = media de la clase positiva mayor que la clase negativa.',
            'Si faltan IDs reales, se alinea por condition × age y orden interno. Los resultados maxT/globales requieren que ese orden corresponda a los mismos ratones.',
        ],
    })
    config = pd.DataFrame([
        {'parameter': 'RUN_PROFILE', 'value': RUN_PROFILE},
        {'parameter': 'N_PERMUTATIONS', 'value': N_PERMUTATIONS},
        {'parameter': 'N_BOOTSTRAP_EFFECT', 'value': N_BOOTSTRAP_EFFECT},
        {'parameter': 'PERMUTATION_BATCH_SIZE', 'value': PERMUTATION_BATCH_SIZE},
        {'parameter': 'RUN_TAG', 'value': RUN_TAG},
        {'parameter': 'RANDOM_STATE', 'value': RANDOM_STATE},
        {'parameter': 'ALPHA', 'value': ALPHA},
        {'parameter': 'CV_SCHEME', 'value': CV_SCHEME},
        {'parameter': 'N_SPLITS', 'value': N_SPLITS},
        {'parameter': 'MODEL_POLICY', 'value': MODEL_POLICY},
        {'parameter': 'SVM_KERNEL', 'value': SVM_KERNEL},
        {'parameter': 'SVM_C', 'value': SVM_C},
        {'parameter': 'INCLUDE_SIMPLE_EFFECTS', 'value': INCLUDE_SIMPLE_EFFECTS},
        {'parameter': 'ALLOW_POSITIONAL_IDS_2D', 'value': ALLOW_POSITIONAL_IDS_2D},
        {'parameter': 'SELECTED_STIMULUS_TYPES', 'value': str(SELECTED_STIMULUS_TYPES)},
        {'parameter': 'SELECTED_ANALYSES', 'value': str(SELECTED_ANALYSES)},
        {'parameter': 'SELECTED_CONTRASTS', 'value': str(SELECTED_CONTRASTS)},
    ])

    with pd.ExcelWriter(path, engine='xlsxwriter') as writer:
        sheets: dict[str, pd.DataFrame] = {
            'All_tests': results,
            'Method_summary': method_summary,
            'Best_candidates': ranking,
            'Effect_sizes': effects,
            'Data_quality': quality,
            'Configuration': config,
            'Method_notes': notes,
        }
        for stimulus_type, sheet_name in {
            '1D_deterministic': '1D_det',
            '1D_stochastic': '1D_stoch',
            '2D_stochastic': '2D_stoch',
        }.items():
            sheets[sheet_name] = results.loc[results['stimulus_type'] == stimulus_type]

        workbook = writer.book
        header_format = workbook.add_format({
            'bold': True, 'font_color': 'white', 'bg_color': '#1F4E78',
            'border': 1, 'align': 'center', 'valign': 'vcenter',
        })
        percent_format = workbook.add_format({'num_format': '0.0%'})
        p_format = workbook.add_format({'num_format': '0.0000'})
        wrap_format = workbook.add_format({'text_wrap': True, 'valign': 'top'})

        for sheet_name, frame in sheets.items():
            frame = _excel_safe_frame(frame)
            frame.to_excel(writer, sheet_name=sheet_name, index=False)
            worksheet = writer.sheets[sheet_name]
            worksheet.freeze_panes(1, 0)
            worksheet.autofilter(0, 0, max(len(frame), 1), max(len(frame.columns) - 1, 0))
            worksheet.set_row(0, 30, header_format)

            for col_idx, column in enumerate(frame.columns):
                values = frame[column].astype(str) if len(frame) else pd.Series(dtype=str)
                max_len = max([len(str(column))] + values.head(500).map(len).tolist())
                width = min(max(max_len + 2, 11), 38)
                fmt = wrap_format if width >= 30 else None
                if any(token in column.lower() for token in ('accuracy', 'threshold', 'chance')):
                    fmt = percent_format
                elif column.lower().startswith('p_') or column.lower().endswith('_p'):
                    fmt = p_format
                worksheet.set_column(col_idx, col_idx, width, fmt)

            # Resalta significancia y desempeño alto cuando las columnas existen.
            if len(frame):
                for p_column in [
                    'p_maxT_fwer_within_analysis',
                    'p_fdr_bh_primary_family',
                    'global_any_p_fdr_bh',
                ]:
                    if p_column in frame.columns:
                        col = frame.columns.get_loc(p_column)
                        worksheet.conditional_format(
                            1, col, len(frame), col,
                            {'type': 'cell', 'criteria': '<=', 'value': ALPHA,
                             'format': workbook.add_format({'bg_color': '#C6EFCE', 'font_color': '#006100'})}
                        )
                if 'balanced_accuracy' in frame.columns:
                    col = frame.columns.get_loc('balanced_accuracy')
                    worksheet.conditional_format(
                        1, col, len(frame), col,
                        {'type': '3_color_scale', 'min_color': '#F8696B',
                         'mid_color': '#FFEB84', 'max_color': '#63BE7B'}
                    )

    print(f'Excel guardado en: {path.resolve()}')


## 4. Validación, tareas y ejecución por etapas

La unidad de checkpoint es **tipo de estímulo × análisis × contraste**. Esto permite detener el proceso, revisar resultados parciales y continuar sin recalcular las tareas ya completadas.

In [ ]:
file_status = validate_project_files(STIMULUS_SETS)
display(file_status)


In [ ]:
# Descubrimiento de features antes de ejecutar permutaciones.
discovery_rows = []
for cfg in STIMULUS_SETS:
    for analysis, file_spec in cfg.files.items():
        try:
            df, groups, source_paths = load_analysis_table(cfg, analysis, file_spec)
        except FileNotFoundError:
            continue
        for group in groups:
            discovery_rows.append({
                'stimulus_type': cfg.stimulus_type,
                'analysis': analysis,
                'stimulus_id': group.stimulus_id,
                'feature_columns': list(group.columns),
                'n_subjects_common': len(df),
                'source_files': [path.name for path in source_paths],
            })
feature_discovery = pd.DataFrame(discovery_rows)
display(feature_discovery)


### 4.1 Lista de tareas y estado de checkpoints

Cada fila puede ejecutarse de forma independiente. Los checkpoints del perfil `development` y `final` quedan en carpetas distintas.

In [ ]:
if not file_status['exists'].all():
    raise FileNotFoundError(
        'Faltan archivos de entrada. Revise PROJECT_ROOT y file_status.'
    )
if not file_status['readable'].all():
    raise ValueError(
        'Hay archivos que existen pero no se pueden cargar. Revise la columna error de file_status.'
    )

ANALYSIS_TASKS = build_analysis_tasks()
task_table = pd.DataFrame(
    ANALYSIS_TASKS,
    columns=['stimulus_type', 'analysis', 'contrast'],
)
task_table.insert(0, 'task_index', range(len(task_table)))
display(task_table)
display(checkpoint_status(ANALYSIS_TASKS))
print(f'Perfil: {RUN_PROFILE} | p mínimo resoluble: {1 / (N_PERMUTATIONS + 1):.6g}')


### 4.2 Ejecutar una sola tarea

Cambie `TASK_INDEX_TO_RUN` por el índice deseado y active `RUN_SELECTED_TASK`. La tabla cruda queda disponible en `partial_result`.

In [ ]:
TASK_INDEX_TO_RUN: int | None = None   # ejemplo: 0
RUN_SELECTED_TASK = False
OVERWRITE_SELECTED_TASK = False

if RUN_SELECTED_TASK:
    if TASK_INDEX_TO_RUN is None:
        raise ValueError('Defina TASK_INDEX_TO_RUN antes de ejecutar.')
    stimulus_type, analysis, contrast_name = ANALYSIS_TASKS[TASK_INDEX_TO_RUN]
    partial_result = run_analysis_task(
        stimulus_type,
        analysis,
        contrast_name,
        overwrite=OVERWRITE_SELECTED_TASK,
    )
    display(partial_result['tests_raw'])
    display(partial_result['method_summary_raw'])
else:
    print('No se ejecutó ninguna tarea. Defina un índice y cambie RUN_SELECTED_TASK=True.')


### 4.3 Revisar un checkpoint ya calculado

In [ ]:
TASK_INDEX_TO_INSPECT: int | None = None   # ejemplo: 0

if TASK_INDEX_TO_INSPECT is not None:
    stimulus_type, analysis, contrast_name = ANALYSIS_TASKS[TASK_INDEX_TO_INSPECT]
    checkpoint = _checkpoint_path(stimulus_type, analysis, contrast_name)
    if not checkpoint.exists():
        raise FileNotFoundError(f'La tarea todavía no tiene checkpoint: {checkpoint}')
    inspected_result = joblib.load(checkpoint)
    display(inspected_result['tests_raw'])
    display(inspected_result['effects'])
    display(inspected_result['method_summary_raw'])
else:
    print('Defina TASK_INDEX_TO_INSPECT para revisar un resultado parcial.')


### 4.4 Ejecutar todas las tareas pendientes

Los checkpoints compatibles se omiten automáticamente. Active la bandera solo cuando quiera continuar con el lote completo.

In [ ]:
RUN_ALL_PENDING = False
STOP_ON_FIRST_ERROR = True

if RUN_ALL_PENDING:
    run_pending_tasks(
        ANALYSIS_TASKS,
        overwrite=False,
        stop_on_error=STOP_ON_FIRST_ERROR,
    )
    display(checkpoint_status(ANALYSIS_TASKS))
else:
    print('RUN_ALL_PENDING=False: no se inició el lote completo.')


### 4.5 Combinar, corregir por multiplicidad y exportar

Esta etapa es rápida porque utiliza únicamente los checkpoints. Active `EXPORT_FINAL_RESULTS` cuando todas las tareas aparezcan como `ready=True`.

In [ ]:
EXPORT_FINAL_RESULTS = False

if EXPORT_FINAL_RESULTS:
    (
        results_df,
        effect_sizes_df,
        method_summary_df,
        data_quality_df,
        ranking_df,
        null_payloads,
    ) = combine_partial_results(ANALYSIS_TASKS)

    excel_path = OUTPUT_DIR / f'neuroscience_significance_results_{RUN_TAG}.xlsx'
    export_excel(
        excel_path,
        results_df,
        effect_sizes_df,
        method_summary_df,
        data_quality_df,
        ranking_df,
    )

    if SAVE_NULL_DISTRIBUTIONS:
        arrays = {}
        for payload in null_payloads:
            key = payload['key']
            arrays[f'{key}__null'] = payload['null_matrix']
            arrays[f'{key}__stimulus_ids'] = payload['stimulus_ids']
        np.savez_compressed(
            OUTPUT_DIR / f'permutation_null_distributions_{RUN_TAG}.npz',
            **arrays,
        )

    print(f'Excel guardado en: {excel_path.resolve()}')
else:
    print('EXPORT_FINAL_RESULTS=False: no se combinaron ni exportaron resultados.')


## 5. Resultados principales

Ejecute esta celda después de la exportación final.

In [ ]:
if 'results_df' in globals():
    significant_candidates = results_df.loc[
        results_df['sig_fdr_bh_primary_family']
        | results_df['significant_maxT']
    ].sort_values([
        'stimulus_type', 'contrast',
        'p_maxT_fwer_within_analysis',
        'p_fdr_bh_primary_family',
    ])
    display(significant_candidates)
    display(method_summary_df)
    display(ranking_df.head(30))
else:
    print('Todavía no se han combinado los resultados finales.')


## Interpretación recomendada

- El único nivel de significancia utilizado en todas las decisiones inferenciales es **α = 0.05**.
- **No interpretar solo la exactitud observada.** Con 30–40 individuos pueden aparecer exactitudes altas por azar; el p de permutación es la referencia principal.
- Para afirmar que un **estímulo concreto** es candidato, priorice `p_maxT_fwer_within_analysis ≤ 0.05` y revise además el tamaño de efecto y su intervalo.
- Para afirmar que una **metodología** sirve como separador, use `global_any_p_fdr_bh` (al menos un estímulo) o `global_consistency_p_fdr_bh` (consistencia promedio).
- `p_fdr_bh_primary_family` responde a la selección exploratoria entre todos los pares metodología–estímulo de una pregunta biológica. `p_fdr_by_primary_family` es una alternativa más conservadora.
- Cohen *d* es descriptivo; con estas muestras se recomienda reportar también **Hedges g**, que corrige el sesgo de muestra pequeña. Para Hx/Hy se reportan efectos por eje y una distancia de Mahalanobis regularizada.
- Un resultado pooled de genotipo o edad no reemplaza el estudio de una interacción. Si hay una hipótesis de interacción, active `INCLUDE_SIMPLE_EFFECTS` o modele explícitamente el diseño factorial.
- Si los hiperparámetros de los SVM se eligieron mirando estas mismas observaciones, el análisis confirmatorio debe repetir esa selección dentro de cada fold y permutación.
